In [0]:
from pyspark.sql import functions as F

silver = spark.table("debiprasad_olist_silver_orders")

In [0]:
gold_category = (
    silver
    .filter(F.col("product_category_name_english").isNotNull())
    .groupBy("product_category_name_english")
    .agg(
        F.countDistinct("order_id").alias("orders"),
        F.avg("delivery_days").alias("avg_delivery_days"),
        F.avg("review_score").alias("avg_review_score"),
        F.avg("is_late").alias("late_rate")
    )
    .orderBy(F.desc("late_rate"))
)

gold_category.show(20, truncate=False)

+---------------------------------+------+------------------+------------------+-------------------+
|product_category_name_english    |orders|avg_delivery_days |avg_review_score  |late_rate          |
+---------------------------------+------+------------------+------------------+-------------------+
|home_comfort_2                   |24    |14.533333333333333|3.6296296296296298|0.16666666666666666|
|furniture_mattress_and_upholstery|38    |14.405405405405405|3.8157894736842106|0.13157894736842105|
|audio                            |350   |13.267955801104973|3.8319444444444444|0.12637362637362637|
|fashion_underwear_beach          |121   |13.606299212598426|3.976923076923077 |0.12213740458015267|
|christmas_supplies               |128   |15.673333333333334|4.02054794520548  |0.11764705882352941|
|books_technical                  |260   |10.577946768060837|4.363636363636363 |0.10861423220973783|
|home_confort                     |397   |13.461538461538462|3.8333333333333335|0.101382488

In [0]:
customer_orders = (
    silver
    .groupBy("customer_unique_id", "order_id", "order_quarter")
    .agg(F.sum("revenue").alias("order_revenue"))
)

customer_counts = (
    customer_orders
    .groupBy("customer_unique_id")
    .agg(F.countDistinct("order_id").alias("order_count"))
)

gold_customer = (
    customer_orders
    .join(customer_counts, "customer_unique_id", "left")
    .groupBy("order_quarter")
    .agg(
        F.countDistinct("customer_unique_id").alias("unique_customers"),
        F.sum("order_revenue").alias("total_revenue"),
        F.countDistinct(
            F.when(F.col("order_count") > 1, F.col("customer_unique_id"))
        ).alias("repeat_customers")
    )
    .withColumn(
        "repeat_customer_share",
        F.col("repeat_customers") / F.col("unique_customers")
    )
    .orderBy("order_quarter")
)

gold_customer.show(truncate=False)

+-------------+----------------+------------------+----------------+---------------------+
|order_quarter|unique_customers|total_revenue     |repeat_customers|repeat_customer_share|
+-------------+----------------+------------------+----------------+---------------------+
|2016-Q3      |4               |354.75            |0               |0.0                  |
|2016-Q4      |322             |56828.46000000002 |13              |0.040372670807453416 |
|2017-Q1      |5153            |855517.7000000031 |262             |0.05084416844556569  |
|2017-Q2      |9139            |1501576.2300000107|531             |0.05810263705000547  |
|2017-Q3      |12335           |1973657.8900000292|700             |0.05674908796108634  |
|2017-Q4      |17505           |2814076.7399999825|803             |0.04587260782633533  |
|2018-Q1      |20767           |3250689.8299999647|962             |0.04632349400491164  |
|2018-Q2      |19715           |3333238.7999999477|763             |0.03870149632259701  |

In [0]:
gold_seller = (
    silver
    .groupBy("seller_id")
    .agg(
        F.countDistinct("order_id").alias("orders"),
        F.avg("delivery_days").alias("avg_delivery_days"),
        F.avg("is_late").alias("late_rate"),
        F.avg("review_score").alias("avg_review_score"),
        F.sum("revenue").alias("total_revenue")
    )
    .filter(F.col("orders") >= 10)
    .orderBy(F.desc("late_rate"))
)

gold_seller.show(20, truncate=False)

+--------------------------------+------+------------------+-------------------+------------------+------------------+
|seller_id                       |orders|avg_delivery_days |late_rate          |avg_review_score  |total_revenue     |
+--------------------------------+------+------------------+-------------------+------------------+------------------+
|b1b3948701c5c72445495bd161b83a4c|18    |23.357142857142858|0.5                |1.7222222222222223|25185.49          |
|2709af9587499e95e803a6498a5a56e9|26    |12.73913043478261 |0.48936170212765956|2.5652173913043477|1859.0199999999998|
|c37b2059d4f90d4feead554e5246565e|12    |19.875            |0.4375             |3.75              |1061.28           |
|f76a3b1349b6df1ee875d1f3fa4340f0|24    |16.875            |0.375              |3.4583333333333335|4748.98           |
|02d35243ea2e497335cd0f076b45675d|14    |25.75             |0.375              |2.6875            |6399.95           |
|c990d6cf976a5718aaedc539f383ab88|10    |21.2222

In [0]:
gold_category.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("debiprasad_olist_gold_category_delivery")

gold_customer.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("debiprasad_olist_gold_customer_revenue")

gold_seller.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("debiprasad_olist_gold_seller_reliability")

print("Gold tables saved")

Gold tables saved
